# 07 — EfficientNet-B3 (transfer learning)

**Pipeline** (shared across all six CNNs):

- **Input:** lesion-cropped images from `X_all.npy` (448×448 storage,
  Otsu segmentation applied in 00_data_setup).
- **Loss:** Focal Loss (γ=2.0) with class-balanced α (Cui et al. 2019, β=0.999).
- **Sampler:** WeightedRandomSampler — every batch is approximately 50/50
  melanoma/non-melanoma despite the 1:8 dataset imbalance.
- **Augmentation (conservative, medical-grade):** RandomResizedCrop(0.85, 1.0)
  + HFlip + VFlip + Rotation(±15°) + mild ColorJitter
  (brightness/contrast=0.10, saturation=0.05, hue=0.02) + small RandomErasing.
  RandAugment, Mixup and CutMix are intentionally disabled because they would
  distort or replace the lesion pixels that the ABCD diagnostic rule depends on.
- **Optimizer:** AdamW (wd=1e-4) with discriminative LR (head 3e-4, backbone 3e-5).
- **Schedule:** linear warmup (3 epochs) → cosine annealing to 1e-6.
- **Stage 1** (head only, 3 epochs, lr=1e-3) → **Stage 2** (full backbone,
  up to 25 epochs, early stopping on val F1, patience=7).
- **EMA** of weights (decay=0.999) — validation, threshold selection and TTA
  all use EMA weights.
- **Test-time augmentation:** 8-way (identity, hflip, vflip, hvflip,
  rot90/180/270, hflip+rot90); probabilities averaged.
- **Threshold:** F1-maximising threshold tuned on validation, applied once on test.

**Disconnect-proof:** the best-val-F1 weights are persisted to
`MyDrive/melanoma/checkpoints/efficientnet_b3_best.pt` every time val F1 improves.
If the runtime dies, the recovery cell at the bottom of this notebook reloads
the checkpoint, runs threshold tuning + TTA evaluation, and saves outputs —
no retraining required.

In [ ]:
# --- Colab setup: ensure the project is on sys.path, mount Drive, load config ---
import os, sys, subprocess
from pathlib import Path

# Either the project is already on disk (uploaded zip / mounted Drive) or we
# clone it from GitHub. We never destroy local changes.
REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
REPO_REF = "codex/clean-data-protocol"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(project_root)], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Mount Drive (silently re-uses an existing mount on re-run)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

# Corrected, versioned artifacts. Do not point this run at legacy data/results.
os.environ["MELANOMA_DATA_DIR"] = "/content/drive/MyDrive/melanoma/data_clean_v2"
os.environ["MELANOMA_RESULTS_DIR"] = "/content/drive/MyDrive/melanoma/results_clean_v2"
os.environ["MELANOMA_CHECKPOINT_DIR"] = "/content/drive/MyDrive/melanoma/checkpoints_clean_v2"
os.environ["MELANOMA_PAPER_DIR"] = "/content/drive/MyDrive/melanoma/paper_clean_v2"
os.environ["MELANOMA_LOCAL_CACHE"] = "/content/local_data_clean_v2"

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Drive root  :", config.DRIVE_ROOT)
print("Data dir    :", config.DATA_DIR)
print("Results dir :", config.RESULTS_DIR)


In [ ]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [ ]:
!pip install --quiet timm 2>&1 | tail -n 1

In [ ]:
# --- Architecture configuration ---
ARCH = "efficientnet_b3"
INPUT_SIZE = config.ARCH_CONFIG[ARCH]["input_size"]
BATCH_SIZE = config.ARCH_CONFIG[ARCH]["batch_size"]
print(f"Architecture: {ARCH}  input={INPUT_SIZE}  batch_size={BATCH_SIZE}")

In [ ]:
# --- Build datasets / dataloaders (data auto-cached to /content/local_data) ---
import numpy as np, torch
from torch.utils.data import DataLoader
from src.data import (load_arrays_balanced, HAMDataset,
                      make_train_transform_strong, make_eval_transform)
from src.training import make_weighted_sampler

X, y, ids, idx_train, idx_val, idx_test = load_arrays_balanced(config.DATA_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "  X:", X.shape)

train_tf = make_train_transform_strong(
    INPUT_SIZE,
    rotation_deg=config.AUG_ROTATION_DEG,
    brightness=config.AUG_COLOR_BRIGHTNESS,
    contrast=config.AUG_COLOR_CONTRAST,
    saturation=config.AUG_COLOR_SATURATION,
    hue=config.AUG_COLOR_HUE,
    crop_scale_min=config.AUG_CROP_SCALE_MIN,
    crop_scale_max=config.AUG_CROP_SCALE_MAX,
    erasing_p=config.AUG_RANDOM_ERASING_P,
    erasing_scale=config.AUG_RANDOM_ERASING_SCALE,
)
eval_tf  = make_eval_transform(INPUT_SIZE)

train_ds = HAMDataset(X, y, idx_train, train_tf)
val_ds   = HAMDataset(X, y, idx_val,   eval_tf)
test_ds  = HAMDataset(X, y, idx_test,  eval_tf)

sampler = make_weighted_sampler(y[idx_train]) if config.USE_WEIGHTED_SAMPLER else None

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE,
                      sampler=sampler, shuffle=(sampler is None),
                      num_workers=2, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
print(f"Batches: train={len(train_ld)}  val={len(val_ld)}  test={len(test_ld)}")

In [ ]:
# --- Build model ---
from src.models import (BUILDERS, freeze_backbone, unfreeze_all,
                        trainable_params, discriminative_param_groups,
                        gradcam_target_layer)

model = BUILDERS[ARCH](num_classes=2, pretrained=True).to(device)
freeze_backbone(model)
print(f"Stage-1 trainable params: {trainable_params(model):,}")

In [ ]:
# --- Focal loss (class-balanced alpha) ---
from src.training import build_focal_loss
criterion = build_focal_loss(y[idx_train],
                             gamma=config.FOCAL_GAMMA,
                             beta=config.FOCAL_BETA,
                             device=device)
print(f"Focal loss: gamma={config.FOCAL_GAMMA}  beta={config.FOCAL_BETA}")
print(f"Class-balanced alpha: {criterion.alpha.cpu().numpy()}")

In [ ]:
# --- Stage 1: head only ---
import time, torch
from src.training import train_loop_v2, EpochLog, EMA, WarmupCosineSchedule

CHECKPOINT_PATH = config.CHECKPOINT_DIR / f"{ARCH}_best.pt"
print("Best weights -> ", CHECKPOINT_PATH)

# EMA shadow tracks the live model from the start
ema = EMA(model, decay=config.CNN_EMA_DECAY)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.CNN_HEAD_LR, weight_decay=config.CNN_WEIGHT_DECAY,
)
log = EpochLog()
t0 = time.time()
log = train_loop_v2(
    model, train_ld, val_ld, criterion, opt, device,
    epochs=config.CNN_HEAD_EPOCHS,
    num_classes=2,
    ema=ema,
    # No mixup/cutmix in stage 1 — head needs clean signal
    mixup_alpha=0.0, cutmix_alpha=0.0, mixup_prob=0.0, cutmix_prob=0.0,
    log=log, checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
# --- Stage 2: unfreeze full backbone with discriminative LR + warmup-cosine ---
unfreeze_all(model)
print(f"Stage-2 trainable params: {trainable_params(model):,}")

groups = discriminative_param_groups(
    model,
    head_lr=config.CNN_FT_LR_HEAD,
    backbone_lr=config.CNN_FT_LR_BACKBONE,
    weight_decay=config.CNN_WEIGHT_DECAY,
)
opt = torch.optim.AdamW(groups)
sched = WarmupCosineSchedule(
    opt,
    warmup_epochs=config.CNN_WARMUP_EPOCHS,
    total_epochs=config.CNN_FT_EPOCHS,
    base_lrs=[config.CNN_FT_LR_BACKBONE, config.CNN_FT_LR_HEAD],
    min_lr=1e-6,
)

log = train_loop_v2(
    model, train_ld, val_ld, criterion, opt, device,
    epochs=config.CNN_FT_EPOCHS,
    num_classes=2,
    scheduler=sched, ema=ema,
    mixup_alpha=config.MIXUP_ALPHA, cutmix_alpha=config.CUTMIX_ALPHA,
    mixup_prob=config.MIXUP_PROB, cutmix_prob=config.CUTMIX_PROB,
    early_stop_patience=config.CNN_EARLY_STOP_PATIENCE,
    log=log, checkpoint_path=CHECKPOINT_PATH,
)
train_time = time.time() - t0
print(f"Total training time: {train_time:.1f}s. Best weights at {CHECKPOINT_PATH}")

In [ ]:
# --- Plot loss / F1 curves and save the raw arrays for the aggregation overlay ---
import numpy as np, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(log.train_loss, label="train"); ax[0].plot(log.val_loss, label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(log.val_f1); ax[1].set_title("Val F1"); ax[1].set_xlabel("epoch")
fig.tight_layout()
fig.savefig(config.RESULTS_DIR / f"{ARCH}_curves.png", dpi=120)
np.savez(config.RESULTS_DIR / f"{ARCH}_curves.npz",
         train_loss=np.array(log.train_loss),
         val_loss=np.array(log.val_loss),
         val_f1=np.array(log.val_f1))
plt.show()

In [ ]:
# --- Apply EMA weights for evaluation ---
ema.apply_shadow(model)

In [ ]:
# --- Single-pass val probabilities (for threshold tuning + 'no TTA' ablation) ---
import numpy as np
from src.training import predict, tune_threshold

y_val_true, _, y_val_prob = predict(model, val_ld, device)
best_t, best_val_f1 = tune_threshold(y_val_true, y_val_prob)
print(f"Threshold (no TTA) best on val: t={best_t:.3f}  F1={best_val_f1:.4f}")

In [ ]:
# --- TTA val probabilities + threshold ---
from src.training import tta_predict
y_val_true_tta, y_val_prob_tta = tta_predict(model, val_ld, device, config.TTA_TRANSFORMS)
best_t_tta, best_val_f1_tta = tune_threshold(y_val_true_tta, y_val_prob_tta)
print(f"Threshold (TTA)   best on val: t={best_t_tta:.3f}  F1={best_val_f1_tta:.4f}")

In [ ]:
# --- Test evaluation: single-pass + TTA, with both thresholds ---
import time
import numpy as np

# Single-pass test
t0 = time.time()
y_test_true, _, y_test_prob = predict(model, test_ld, device)
inf_ms_single = (time.time() - t0) * 1000.0 / len(y_test_true)

# TTA test
t0 = time.time()
y_test_true2, y_test_prob_tta = tta_predict(model, test_ld, device, config.TTA_TRANSFORMS)
inf_ms_tta = (time.time() - t0) * 1000.0 / len(y_test_true2)
assert np.array_equal(y_test_true, y_test_true2)

y_pred_single = (y_test_prob > best_t).astype(int)
y_pred_tta    = (y_test_prob_tta > best_t_tta).astype(int)

print(f"Single-pass inf: {inf_ms_single:.2f} ms/img")
print(f"TTA (8-way) inf: {inf_ms_tta:.2f} ms/img")

In [ ]:
# --- Save standardized outputs (TTA result is the headline, but both columns persist) ---
import pandas as pd
from src.evaluation import save_standard_outputs, compute_metrics

# Headline metrics use TTA + tuned threshold
hp = dict(
    arch=ARCH,
    input_size=int(INPUT_SIZE),
    batch_size=int(BATCH_SIZE),
    head_lr=config.CNN_HEAD_LR,
    head_epochs=config.CNN_HEAD_EPOCHS,
    ft_lr_head=config.CNN_FT_LR_HEAD,
    ft_lr_backbone=config.CNN_FT_LR_BACKBONE,
    ft_epochs=config.CNN_FT_EPOCHS,
    warmup_epochs=config.CNN_WARMUP_EPOCHS,
    early_stop_patience=config.CNN_EARLY_STOP_PATIENCE,
    weight_decay=config.CNN_WEIGHT_DECAY,
    ema_decay=config.CNN_EMA_DECAY,
    focal_gamma=config.FOCAL_GAMMA,
    focal_beta=config.FOCAL_BETA,
    weighted_sampler=bool(config.USE_WEIGHTED_SAMPLER),
    augmentation=("RandomResizedCrop(0.85,1.0)+HFlip+VFlip+"
                  f"Rotation({config.AUG_ROTATION_DEG})+mildColorJitter+"
                  f"RandomErasing(p={config.AUG_RANDOM_ERASING_P}); "
                  "no Mixup/CutMix/RandAugment"),
    stored_image_size=int(config.IMG_SIZE),
    segmentation="Otsu(LAB-L invert)+morph close/open+largest CC+15% margin crop",
    split="lesion-grouped stratified 70/15/15",
    decision_threshold_tta=best_t_tta,
    decision_threshold_single=best_t,
    threshold_selection="argmax F1 on validation set",
    tta_transforms=list(config.TTA_TRANSFORMS),
)

metrics = save_standard_outputs(
    method_name=ARCH,
    results_dir=config.RESULTS_DIR,
    y_true=y_test_true,
    y_pred=y_pred_tta,         # headline = TTA + tuned threshold
    y_prob=y_test_prob_tta,    # headline probability = TTA-averaged
    ids=ids[idx_test],
    hyperparameters=hp,
    train_time_sec=train_time,
    inference_time_per_image_ms=inf_ms_tta,
)

# Also save the rich TEST predictions CSV with both single and TTA columns
# (the aggregation notebook needs this to compute the TTA ablation row).
pd.DataFrame({
    "image_id": ids[idx_test],
    "y_true":   y_test_true,
    "y_pred_single": y_pred_single,
    "y_prob_single": y_test_prob,
    "y_pred_tta":    y_pred_tta,
    "y_prob_tta":    y_test_prob_tta,
    "best_t_single": best_t,
    "best_t_tta":    best_t_tta,
}).to_csv(config.RESULTS_DIR / f"{ARCH}_predictions.csv", index=False)

# VAL predictions CSV — required by the ensemble (method 9) for threshold
# tuning on a held-out set. The aggregation notebook averages these
# probabilities across all 6 CNNs, sweeps thresholds on val, and applies
# the best threshold once on the test ensemble.
pd.DataFrame({
    "image_id": ids[idx_val],
    "y_true":   y_val_true,
    "y_prob_single": y_val_prob,
    "y_prob_tta":    y_val_prob_tta,
}).to_csv(config.RESULTS_DIR / f"{ARCH}_val_predictions.csv", index=False)

# Side-by-side diagnostic
m_single = compute_metrics(y_test_true, y_pred_single, y_test_prob)
m_tta    = compute_metrics(y_test_true, y_pred_tta,    y_test_prob_tta)
print("Single-pass test metrics:", {k: round(v, 4) for k, v in m_single.items()})
print("TTA-headline   test metrics:", {k: round(v, 4) for k, v in m_tta.items()})

In [ ]:
# --- Grad-CAM grid: 4 melanoma + 4 non-melanoma test images ---
import numpy as np, torch, matplotlib.pyplot as plt
from src.gradcam import GradCAM, overlay_heatmap

target_layer = gradcam_target_layer(model, ARCH)
cam = GradCAM(model, target_layer)
mel_idx    = idx_test[y[idx_test] == 1][:4]
nonmel_idx = idx_test[y[idx_test] == 0][:4]
chosen = list(mel_idx) + list(nonmel_idx)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, k in zip(axes.flat, chosen):
    img = X[k]
    t = eval_tf(img).unsqueeze(0).to(device)
    try:
        heat = cam(t, class_idx=int(y[k]))
        ax.imshow(overlay_heatmap(img, heat))
    except Exception:
        # Some attention archs (e.g. Swin) may not produce a valid CAM via
        # plain gradcam; fall back to showing the raw image.
        ax.imshow(img); ax.set_title(f"Grad-CAM N/A for {ARCH}")
    ax.set_title(f"id={ids[k]}  true={int(y[k])}")
    ax.axis("off")
cam.close()
fig.tight_layout()
fig.savefig(config.RESULTS_DIR / f"{ARCH}_gradcam_grid.png", dpi=120)
plt.show()

---
## Recovery cell (use only if the runtime disconnected during training)

If the cells above never finished but you saw at least one
`-> checkpoint saved` line during training, the best weights are at
`MyDrive/melanoma/checkpoints/efficientnet_b3_best.pt`. Run **only the recovery cell
below** on a fresh runtime (after re-running the preamble + dataset cells)
to finish evaluation without retraining.

In [ ]:
# --- Recovery: load checkpoint, re-run threshold + TTA, save outputs ---
import torch, time, numpy as np, pandas as pd
from src.training import predict, tune_threshold, tta_predict
from src.evaluation import save_standard_outputs

CHECKPOINT_PATH = config.CHECKPOINT_DIR / f"{ARCH}_best.pt"
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

y_val_true, _, y_val_prob = predict(model, val_ld, device)
best_t, _ = tune_threshold(y_val_true, y_val_prob)
y_val_true_tta, y_val_prob_tta = tta_predict(model, val_ld, device, config.TTA_TRANSFORMS)
best_t_tta, _ = tune_threshold(y_val_true_tta, y_val_prob_tta)

t0 = time.time()
y_test_true, _, y_test_prob = predict(model, test_ld, device)
inf_ms_single = (time.time() - t0) * 1000.0 / len(y_test_true)
t0 = time.time()
_, y_test_prob_tta = tta_predict(model, test_ld, device, config.TTA_TRANSFORMS)
inf_ms_tta = (time.time() - t0) * 1000.0 / len(y_test_true)

y_pred_tta = (y_test_prob_tta > best_t_tta).astype(int)
y_pred_single = (y_test_prob > best_t).astype(int)
metrics = save_standard_outputs(
    method_name=ARCH,
    results_dir=config.RESULTS_DIR,
    y_true=y_test_true, y_pred=y_pred_tta, y_prob=y_test_prob_tta,
    ids=ids[idx_test],
    hyperparameters={"recovered_from_checkpoint": True,
                     "decision_threshold_tta": best_t_tta,
                     "decision_threshold_single": best_t},
    train_time_sec=0.0,
    inference_time_per_image_ms=inf_ms_tta,
)

# Save the rich predictions CSVs (test + val) — same layout as the main
# pipeline cell so the aggregation notebook can read either source.
pd.DataFrame({
    "image_id": ids[idx_test],
    "y_true":   y_test_true,
    "y_pred_single": y_pred_single,
    "y_prob_single": y_test_prob,
    "y_pred_tta":    y_pred_tta,
    "y_prob_tta":    y_test_prob_tta,
    "best_t_single": best_t,
    "best_t_tta":    best_t_tta,
}).to_csv(config.RESULTS_DIR / f"{ARCH}_predictions.csv", index=False)
pd.DataFrame({
    "image_id": ids[idx_val],
    "y_true":   y_val_true,
    "y_prob_single": y_val_prob,
    "y_prob_tta":    y_val_prob_tta,
}).to_csv(config.RESULTS_DIR / f"{ARCH}_val_predictions.csv", index=False)
{k: round(v, 4) for k, v in metrics.items() if isinstance(v, (int, float))}